# 01 · Build the grid-year panel from Google Earth Engine

This notebook is the first real data notebook in the workflow. It starts from the actual source of the raw data: **Google Earth Engine (GEE)**.

## Goals of this notebook
1. Connect to Earth Engine.
2. Load the grid geometry.
3. Pull annual deforestation information for each grid cell.
4. Export the extracted tables and the grid geometry.
5. Read the exported files back locally.
6. Validate the grid-year panel.
7. Produce the first descriptive graphs and maps.
8. Save the local panel files that downstream notebooks will use.

## Why this notebook is structured in two stages
For small objects, data can sometimes be pulled directly from Earth Engine into Python. For a full grid-level panel, that is often too heavy. A more robust pattern is:

- use **Earth Engine** to compute the zonal statistics,
- export the results as tables,
- then read and validate those exported files locally.

This notebook follows exactly that pattern.

In [ ]:
# Optional installation cell for exact replication in the CURRENT notebook environment.
# Run this only if you need to install or update packages.

from pathlib import Path
import sys

req = (Path.cwd().resolve().parent / "requirements.txt").resolve()
print("Using requirements file:", req)
print("Current Python executable:", sys.executable)

Using requirements file: C:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2\requirements.txt
Current Python executable: c:\Users\cpedr\OneDrive - Hertie School\PhD\Paper 2\paper2\.venv\Scripts\python.exe


In [8]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Imports and environment report

This notebook is self-contained. All helper functions used at this stage are defined below instead of being imported from local modules.

In [9]:
from __future__ import annotations

from pathlib import Path
import importlib.metadata as ilm
import json
import os
import re
import warnings
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd

try:
    import ee
except Exception:
    ee = None

try:
    import geemap
except Exception:
    geemap = None

In [ ]:
print("Python executable:", os.sys.executable)
print("Python version:", os.sys.version)
for pkg in ["numpy", "pandas", "matplotlib", "geopandas", "shapely", "pyproj", "pyogrio", "earthengine-api", "geemap"]:
    try:
        print(f"{pkg}: {ilm.version(pkg)}")
    except Exception:
        print(f"{pkg}: not installed")

## Detect the project root

The notebook first tries to infer the project root from the current working directory.
If that fails in your local setup, set `MANUAL_PROJECT_ROOT` to the absolute path of the project folder.

In [ ]:
MANUAL_PROJECT_ROOT = None
# Example for Windows:
# MANUAL_PROJECT_ROOT = r"C:\Users\YOUR_NAME\OneDrive - Hertie School\PhD\Paper 2\paper2"


def detect_project_root(manual_root=None):
    if manual_root is not None:
        return Path(manual_root).expanduser().resolve()

    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for candidate in candidates:
        if (candidate / "notebooks").exists():
            return candidate
    return cwd


PROJECT_ROOT = detect_project_root(MANUAL_PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd().resolve())

## Helper functions

In [ ]:
def ensure_project_structure(project_root: Path):
    folders = [
        project_root / "notebooks",
        project_root / "configs",
        project_root / "data" / "raw",
        project_root / "data" / "raw" / "gee_exports",
        project_root / "data" / "intermediate",
        project_root / "data" / "processed",
        project_root / "outputs" / "figures",
        project_root / "outputs" / "maps",
        project_root / "outputs" / "tables",
        project_root / "logs",
    ]
    for folder in folders:
        folder.mkdir(parents=True, exist_ok=True)
    return folders


def set_plot_theme():
    plt.style.use("default")
    plt.rcParams.update({
        "figure.figsize": (10, 6),
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "figure.dpi": 120,
        "savefig.bbox": "tight",
    })


def save_figure(fig, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=200)
    print("Saved figure:", path)


def read_vector_any(path, layer=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Vector file not found: {path}")
    if layer is not None:
        return gpd.read_file(path, layer=layer)
    return gpd.read_file(path)


def standardize_grid_gdf(gdf: gpd.GeoDataFrame, grid_id_col: str) -> gpd.GeoDataFrame:
    gdf = gdf.copy()
    if grid_id_col not in gdf.columns:
        raise ValueError(f"The grid id column '{grid_id_col}' is not present in the geometry file.")
    gdf[grid_id_col] = gdf[grid_id_col].astype(str)
    if gdf.crs is None:
        warnings.warn("The grid geometry has no CRS. Assuming EPSG:4326 for Earth Engine compatibility.")
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")
    return gdf


def try_initialize_ee(project_id: str):
    if ee is None:
        raise ModuleNotFoundError("earthengine-api is not installed in the current environment.")
    try:
        ee.Initialize(project=project_id)
        print("Earth Engine initialized successfully.")
    except Exception as exc:
        print("Initial ee.Initialize() failed.")
        print("Attempting interactive authentication...")
        ee.Authenticate()
        ee.Initialize(project=project_id)
        print("Earth Engine authenticated and initialized successfully.")


def make_grid_feature_collection(grid_source: str, grid_asset_id: str | None, local_grid_path: Path | None, local_grid_layer: str | None, grid_id_col: str):
    local_gdf = None
    if grid_source == "ee_asset":
        if not grid_asset_id:
            raise ValueError("GRID_ASSET_ID must be provided when GRID_SOURCE='ee_asset'.")
        fc = ee.FeatureCollection(grid_asset_id)
        return fc, local_gdf

    if grid_source == "local_file":
        if geemap is None:
            raise ModuleNotFoundError("geemap is required to convert a local GeoDataFrame to an Earth Engine FeatureCollection.")
        if local_grid_path is None:
            raise ValueError("LOCAL_GRID_PATH must be provided when GRID_SOURCE='local_file'.")
        local_gdf = read_vector_any(local_grid_path, layer=local_grid_layer)
        local_gdf = standardize_grid_gdf(local_gdf, grid_id_col)
        fc = geemap.gdf_to_ee(local_gdf)
        return fc, local_gdf

    raise ValueError("GRID_SOURCE must be either 'ee_asset' or 'local_file'.")


def hansen_annual_loss_area_image(year: int, dataset_id: str, treecover_threshold: int = 30):
    if year < 2001:
        raise ValueError("The Hansen annual loss coding begins in 2001.")
    gfc = ee.Image(dataset_id)
    lossyear = gfc.select("lossyear")
    canopy = gfc.select("treecover2000").gte(treecover_threshold)
    year_code = year - 2000
    return (
        ee.Image.pixelArea()
        .updateMask(lossyear.eq(year_code))
        .updateMask(canopy)
        .rename("loss_area_m2")
    )


def build_annual_loss_feature_collection(grid_fc, year: int, dataset_id: str, treecover_threshold: int, scale_m: int, tile_scale: int = 4):
    img = hansen_annual_loss_area_image(year, dataset_id, treecover_threshold)
    reduced = img.reduceRegions(
        collection=grid_fc,
        reducer=ee.Reducer.sum(),
        scale=scale_m,
        tileScale=tile_scale,
    )

    def _set_props(ft):
        val = ee.Number(ft.get("sum"))
        return ft.set({"year": year, "loss_area_m2": val}).select(ft.propertyNames().add("year").add("loss_area_m2"))

    return reduced.map(_set_props)


def create_drive_export_task(feature_collection, description: str, folder: str, file_name_prefix: str, file_format: str = "CSV", selectors=None):
    task = ee.batch.Export.table.toDrive(
        collection=feature_collection,
        description=description,
        folder=folder,
        fileNamePrefix=file_name_prefix,
        fileFormat=file_format,
        selectors=selectors,
    )
    return task


def parse_year_from_filename(path: Path):
    m = re.search(r"(19|20)\d{2}", path.stem)
    return int(m.group()) if m else None


def combine_loss_exports(local_export_dir: Path, export_prefix: str, grid_id_col: str):
    csv_paths = sorted(local_export_dir.glob(f"{export_prefix}_*.csv"))
    if not csv_paths:
        raise FileNotFoundError(
            f"No CSV exports found in {local_export_dir}. Complete the Earth Engine exports, download the CSV files, and place them in this folder."
        )

    frames = []
    for path in csv_paths:
        df = pd.read_csv(path)
        if grid_id_col not in df.columns:
            raise ValueError(f"{path.name} does not contain the grid id column '{grid_id_col}'.")
        if "year" not in df.columns:
            inferred_year = parse_year_from_filename(path)
            if inferred_year is None:
                raise ValueError(f"Could not infer the year from {path.name} and no 'year' column exists.")
            df["year"] = inferred_year
        rename_map = {"sum": "loss_area_m2"} if "sum" in df.columns and "loss_area_m2" not in df.columns else {}
        df = df.rename(columns=rename_map)
        keep_cols = [c for c in [grid_id_col, "year", "loss_area_m2"] if c in df.columns]
        df = df[keep_cols].copy()
        df[grid_id_col] = df[grid_id_col].astype(str)
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
        df["loss_area_m2"] = pd.to_numeric(df["loss_area_m2"], errors="coerce").fillna(0.0)
        frames.append(df)

    out = pd.concat(frames, ignore_index=True)
    out["loss_area_ha"] = out["loss_area_m2"] / 10000.0
    out = out.sort_values([grid_id_col, "year"]).reset_index(drop=True)
    return out


def validate_panel(panel_df: pd.DataFrame, grid_id_col: str):
    assert grid_id_col in panel_df.columns, f"Missing grid id column: {grid_id_col}"
    assert "year" in panel_df.columns, "Missing year column"
    assert "loss_area_m2" in panel_df.columns, "Missing loss_area_m2 column"
    duplicated = panel_df.duplicated([grid_id_col, "year"]).sum()
    if duplicated > 0:
        raise ValueError(f"The panel contains {duplicated} duplicated grid-year rows.")
    print("Rows:", len(panel_df))
    print("Unique grid ids:", panel_df[grid_id_col].nunique())
    print("Years:", int(panel_df["year"].min()), "to", int(panel_df["year"].max()))
    print("Duplicate grid-year rows:", duplicated)
    return True


def load_grid_geometry_for_mapping(grid_source: str, local_grid_path: Path | None, local_grid_layer: str | None, local_grid_export_path: Path, grid_id_col: str, local_grid_gdf_from_source=None):
    if local_grid_gdf_from_source is not None:
        return standardize_grid_gdf(local_grid_gdf_from_source, grid_id_col)
    if local_grid_path is not None and Path(local_grid_path).exists():
        gdf = read_vector_any(local_grid_path, layer=local_grid_layer)
        return standardize_grid_gdf(gdf, grid_id_col)
    if local_grid_export_path.exists():
        gdf = gpd.read_file(local_grid_export_path)
        return standardize_grid_gdf(gdf, grid_id_col)
    return None


def plot_annual_loss_trend(panel_df: pd.DataFrame, year_col: str = "year", value_col: str = "loss_area_ha"):
    annual = panel_df.groupby(year_col, as_index=False)[value_col].sum()
    fig, ax = plt.subplots()
    ax.plot(annual[year_col], annual[value_col], marker="o")
    ax.set_title("Total deforestation by year across all grid cells")
    ax.set_xlabel("Year")
    ax.set_ylabel("Loss area (hectares)")
    return fig, ax, annual


def plot_loss_histogram(panel_df: pd.DataFrame, value_col: str = "loss_area_ha"):
    fig, ax = plt.subplots()
    ax.hist(panel_df[value_col], bins=40)
    ax.set_title("Distribution of annual deforestation across grid-year observations")
    ax.set_xlabel("Loss area (hectares)")
    ax.set_ylabel("Count")
    return fig, ax


def plot_year_map(grid_gdf: gpd.GeoDataFrame, panel_df: pd.DataFrame, year: int, grid_id_col: str, value_col: str = "loss_area_ha"):
    year_df = panel_df.loc[panel_df["year"] == year, [grid_id_col, value_col]].copy()
    map_gdf = grid_gdf.merge(year_df, on=grid_id_col, how="left")
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    map_gdf.plot(column=value_col, ax=ax, legend=True, missing_kwds={"color": "lightgrey"})
    ax.set_title(f"Grid-level deforestation in {year}")
    ax.set_axis_off()
    return fig, ax, map_gdf

## Configuration

Fill in the project-specific settings below before running the Earth Engine extraction.

### Notes on the default deforestation source
This notebook defaults to the **Hansen Global Forest Change v1.12 (2000–2024)** Earth Engine dataset. The extraction uses the `lossyear` band to identify annual forest loss and multiplies the masked pixels by pixel area to obtain annual loss area per grid cell.

If you want a different deforestation source, this is the section to change. The rest of the notebook can stay the same as long as the Earth Engine extraction produces one row per `grid_id × year`.

In [ ]:
ensure_project_structure(PROJECT_ROOT)
set_plot_theme()

# -----------------------------
# Google Earth Engine settings
# -----------------------------
GEE_PROJECT = "your-google-cloud-project-id"
GRID_SOURCE = "ee_asset"              # "ee_asset" or "local_file"
GRID_ASSET_ID = "projects/your-project/assets/your_grid_asset"
LOCAL_GRID_PATH = None                 # Example: PROJECT_ROOT / "data/raw/grid_geometry.gpkg"
LOCAL_GRID_LAYER = None                # Example: "grid"
GRID_ID_COL = "grid_id"

# -----------------------------
# Deforestation source settings
# -----------------------------
DEFORESTATION_DATASET_ID = "UMD/hansen/global_forest_change_2024_v1_12"
ANALYSIS_YEARS = list(range(2001, 2025))
TREECOVER_THRESHOLD = 30               # percent canopy cover in treecover2000
REDUCTION_SCALE_M = 30
TILE_SCALE = 4

# -----------------------------
# Export settings
# -----------------------------
DRIVE_FOLDER = "gee_exports_paper2"
EXPORT_PREFIX = "grid_loss_hansen"
EXPORT_GRID_GEOMETRY = True
GRID_GEOMETRY_EXPORT_PREFIX = "grid_geometry"
START_EXPORT_TASKS = False             # Set to True only when you are ready to start the EE tasks.

# -----------------------------
# Local file paths
# -----------------------------
LOCAL_EXPORT_DIR = PROJECT_ROOT / "data" / "raw" / "gee_exports"
LOCAL_GRID_EXPORT_PATH = LOCAL_EXPORT_DIR / f"{GRID_GEOMETRY_EXPORT_PREFIX}.geojson"
PANEL_RAW_PATH = PROJECT_ROOT / "data" / "raw" / "panel_raw.parquet"
GRID_GEOMETRY_CLEAN_PATH = PROJECT_ROOT / "data" / "raw" / "grid_geometry_clean.parquet"
PANEL_WITH_GEOMETRY_PATH = PROJECT_ROOT / "data" / "intermediate" / "panel_raw_with_geometry.parquet"
SAMPLE_MAP_YEAR = 2024

LOCAL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

## Authenticate and initialize Earth Engine

Earth Engine requires authentication and initialization in Python. If the first initialization attempt fails, the code below triggers interactive authentication and then retries initialization.

In [ ]:
if GEE_PROJECT == "your-google-cloud-project-id":
    warnings.warn("Replace GEE_PROJECT with your actual Google Cloud project id before running the Earth Engine extraction.")

try_initialize_ee(GEE_PROJECT)

## Load the grid geometry into Earth Engine

There are two supported routes:

- `GRID_SOURCE = "ee_asset"`: the grid already exists as an Earth Engine asset;
- `GRID_SOURCE = "local_file"`: the grid exists locally and is converted to an Earth Engine `FeatureCollection` with `geemap`.

In [ ]:
grid_fc, local_grid_gdf_from_source = make_grid_feature_collection(
    grid_source=GRID_SOURCE,
    grid_asset_id=GRID_ASSET_ID,
    local_grid_path=LOCAL_GRID_PATH,
    local_grid_layer=LOCAL_GRID_LAYER,
    grid_id_col=GRID_ID_COL,
)

first_feature = ee.Feature(grid_fc.first())
property_names = first_feature.propertyNames().getInfo()
print("Grid properties detected in Earth Engine:", property_names)
if GRID_ID_COL not in property_names:
    raise ValueError(
        f"The grid asset does not contain the column '{GRID_ID_COL}'. Update GRID_ID_COL or rebuild the asset."
    )
print("Grid size (number of features):", grid_fc.size().getInfo())

## Optional interactive preview map

This is only a quick visual check. The main extraction happens in the next section.

In [ ]:
if geemap is None:
    print("geemap is not installed, so the interactive preview map is skipped.")
else:
    preview_map = geemap.Map()
    preview_map.centerObject(grid_fc, 6)
    preview_map.addLayer(
        grid_fc.style(**{"color": "cyan", "fillColor": "00000000", "width": 1}),
        {},
        "Grid geometry",
    )
    preview_map.addLayer(
        ee.Image(DEFORESTATION_DATASET_ID).select("lossyear").selfMask(),
        {"min": 1, "max": 24},
        "Hansen loss year",
    )
    display(preview_map)

## Build the Earth Engine export tasks

For each year, the notebook creates a `FeatureCollection` containing the grid id, the calendar year, and the annual loss area in square meters.

By default, the notebook **creates** the export tasks but does **not start** them. This keeps the workflow safe while you are still checking the configuration.

In [ ]:
loss_export_tasks = {}
for year in ANALYSIS_YEARS:
    fc_year = build_annual_loss_feature_collection(
        grid_fc=grid_fc,
        year=year,
        dataset_id=DEFORESTATION_DATASET_ID,
        treecover_threshold=TREECOVER_THRESHOLD,
        scale_m=REDUCTION_SCALE_M,
        tile_scale=TILE_SCALE,
    ).select([GRID_ID_COL, "year", "loss_area_m2"])

    description = f"{EXPORT_PREFIX}_{year}"
    task = create_drive_export_task(
        feature_collection=fc_year,
        description=description,
        folder=DRIVE_FOLDER,
        file_name_prefix=description,
        file_format="CSV",
        selectors=[GRID_ID_COL, "year", "loss_area_m2"],
    )
    if START_EXPORT_TASKS:
        task.start()
    loss_export_tasks[year] = task

print(f"Prepared {len(loss_export_tasks)} annual export tasks.")
print("START_EXPORT_TASKS =", START_EXPORT_TASKS)

## Optional export task for the grid geometry

This export is useful when the grid lives in Earth Engine as an asset and you want a local copy for mapping and merging.

In [ ]:
grid_geometry_task = None
if EXPORT_GRID_GEOMETRY:
    grid_geometry_task = create_drive_export_task(
        feature_collection=grid_fc.select([GRID_ID_COL]),
        description=GRID_GEOMETRY_EXPORT_PREFIX,
        folder=DRIVE_FOLDER,
        file_name_prefix=GRID_GEOMETRY_EXPORT_PREFIX,
        file_format="GeoJSON",
        selectors=[GRID_ID_COL],
    )
    if START_EXPORT_TASKS:
        grid_geometry_task.start()
    print("Prepared grid geometry export task.")
else:
    print("Grid geometry export skipped.")

## Manual step after the export tasks are created

If `START_EXPORT_TASKS = False`, set it to `True` and rerun the export cells when you are ready.

Then:
1. let the Earth Engine tasks complete,
2. download the exported CSV files and the optional GeoJSON file from Google Drive,
3. place them inside `data/raw/gee_exports/`,
4. rerun the notebook from the next section onward.

This two-stage pattern is intentional. It keeps the heavy geospatial computation in Earth Engine and the panel validation in Python.

## Read the local Earth Engine exports back into Python

In [ ]:
exported_files = sorted(LOCAL_EXPORT_DIR.glob("*"))
print("Files currently present in the local export directory:")
for path in exported_files[:20]:
    print(" -", path.name)
if len(exported_files) > 20:
    print(f"... and {len(exported_files) - 20} more")

In [ ]:
panel_df = combine_loss_exports(
    local_export_dir=LOCAL_EXPORT_DIR,
    export_prefix=EXPORT_PREFIX,
    grid_id_col=GRID_ID_COL,
)
validate_panel(panel_df, GRID_ID_COL)
panel_df.head()

## Load the local grid geometry for merging and mapping

The notebook first tries to use a local copy that was already available. If that does not exist, it looks for the Earth Engine export of the grid geometry.

In [ ]:
grid_gdf = load_grid_geometry_for_mapping(
    grid_source=GRID_SOURCE,
    local_grid_path=LOCAL_GRID_PATH,
    local_grid_layer=LOCAL_GRID_LAYER,
    local_grid_export_path=LOCAL_GRID_EXPORT_PATH,
    grid_id_col=GRID_ID_COL,
    local_grid_gdf_from_source=local_grid_gdf_from_source,
)

if grid_gdf is None:
    warnings.warn(
        "No local grid geometry is available yet. You can still save the panel table, but mapping cells will be skipped until a local geometry file is available."
    )
else:
    print(grid_gdf.head())
    print("Grid CRS:", grid_gdf.crs)
    print("Number of grid features:", len(grid_gdf))

## Merge the panel with the geometry and save the local files

In [ ]:
panel_df.to_parquet(PANEL_RAW_PATH, index=False)
print("Saved panel table:", PANEL_RAW_PATH)

if grid_gdf is not None:
    grid_gdf.to_parquet(GRID_GEOMETRY_CLEAN_PATH, index=False)
    panel_with_geom = grid_gdf.merge(panel_df, on=GRID_ID_COL, how="left")
    panel_with_geom.to_parquet(PANEL_WITH_GEOMETRY_PATH, index=False)
    print("Saved grid geometry:", GRID_GEOMETRY_CLEAN_PATH)
    print("Saved panel with geometry:", PANEL_WITH_GEOMETRY_PATH)

## Descriptive tables

This is the first quality-control checkpoint. The goal is to make sure the extracted panel looks structurally correct before moving on to treatment timing and spatial exposure.

In [ ]:
summary_table = pd.DataFrame({
    "metric": [
        "rows",
        "unique_grid_ids",
        "min_year",
        "max_year",
        "total_loss_area_ha",
        "mean_loss_area_ha",
        "median_loss_area_ha",
    ],
    "value": [
        len(panel_df),
        panel_df[GRID_ID_COL].nunique(),
        int(panel_df["year"].min()),
        int(panel_df["year"].max()),
        float(panel_df["loss_area_ha"].sum()),
        float(panel_df["loss_area_ha"].mean()),
        float(panel_df["loss_area_ha"].median()),
    ],
})
summary_table

In [ ]:
summary_table_path = PROJECT_ROOT / "outputs" / "tables" / "01_panel_summary.csv"
summary_table.to_csv(summary_table_path, index=False)
print("Saved:", summary_table_path)

## Graph 1 · Total deforestation by year

In [ ]:
fig, ax, annual_totals = plot_annual_loss_trend(panel_df)
plt.show()
save_figure(fig, PROJECT_ROOT / "outputs" / "figures" / "01_total_deforestation_by_year.png")
annual_totals.head()

## Graph 2 · Distribution of annual loss across grid-year observations

In [ ]:
fig, ax = plot_loss_histogram(panel_df)
plt.show()
save_figure(fig, PROJECT_ROOT / "outputs" / "figures" / "01_histogram_grid_year_loss.png")

## Map 1 · Grid-level deforestation in a selected year

In [ ]:
if grid_gdf is None:
    print("Map skipped because no local grid geometry is available yet.")
else:
    fig, ax, map_gdf = plot_year_map(grid_gdf, panel_df, SAMPLE_MAP_YEAR, GRID_ID_COL)
    plt.show()
    save_figure(fig, PROJECT_ROOT / "outputs" / "maps" / f"01_grid_deforestation_{SAMPLE_MAP_YEAR}.png")

## Map 2 · Data coverage by grid cell

In [ ]:
if grid_gdf is None:
    print("Coverage map skipped because no local grid geometry is available yet.")
else:
    coverage = panel_df.groupby(GRID_ID_COL, as_index=False)["year"].nunique().rename(columns={"year": "n_years_observed"})
    coverage_gdf = grid_gdf.merge(coverage, on=GRID_ID_COL, how="left")
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    coverage_gdf.plot(column="n_years_observed", ax=ax, legend=True, missing_kwds={"color": "lightgrey"})
    ax.set_title("Number of observed years by grid cell")
    ax.set_axis_off()
    plt.show()
    save_figure(fig, PROJECT_ROOT / "outputs" / "maps" / "01_grid_data_coverage.png")

## End-of-notebook checklist

Before moving on to Notebook 02, check that you now have:

- `data/raw/panel_raw.parquet`
- `data/raw/grid_geometry_clean.parquet` (if a local geometry was available)
- `data/intermediate/panel_raw_with_geometry.parquet` (if geometry was available)
- at least one summary table in `outputs/tables/`
- at least one graph in `outputs/figures/`
- and, if geometry was available, at least one map in `outputs/maps/`

Notebook 02 will define treatment timing, first treatment year, cohorts, and event time.